In [ ]:
import os
import re
import time
import hashlib
from typing import Dict, Iterable, List, Optional
import requests
import pandas as pd

STARTGG_API_URL = "https://api.start.gg/gql/alpha"
STARTGG_API_KEY = os.getenv("START_GG_API_KEY") or os.getenv("SMASHGG_TOKEN")

if not STARTGG_API_KEY:
    raise ValueError("Missing START_GG_API_KEY or SMASHGG_TOKEN in environment")

HEADERS = {
    "Authorization": f"Bearer {STARTGG_API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

AUTH_TYPES = ["DISCORD", "TWITTER", "TWITCH"]

def gql(query: str, variables: Optional[Dict] = None, timeout: int = 30) -> Dict:
    payload = {"query": query, "variables": variables or {}}
    response = requests.post(STARTGG_API_URL, headers=HEADERS, json=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    if "errors" in data:
        raise RuntimeError(data["errors"])
    return data

def extract_startgg_path(url_or_slug: str) -> str:
    match = re.search(r"(?:https?://)?(?:www\.)?(?:start|smash)\.gg/(.+)", url_or_slug.strip())
    if match:
        return match.group(1).strip("/")
    return url_or_slug.strip().strip("/")

def is_event_slug(slug: str) -> bool:
    return "/event/" in slug

def normalize_tournament_slug(slug_or_url: str) -> str:
    path = extract_startgg_path(slug_or_url)
    if path.startswith("tournament/"):
        return path
    if path.startswith("event/"):
        return f"tournament/{path}"
    return f"tournament/{path}"

def resolve_event_slug(tournament_slug: str, event_name_contains: Optional[str] = None) -> str:
    query = """
    query TournamentEvents($slug: String) {
      tournament(slug: $slug) {
        events {
          id
          name
          slug
        }
      }
    }
    """
    data = gql(query, {"slug": tournament_slug})
    events = (data.get("data", {}).get("tournament", {}) or {}).get("events", [])
    if not events:
        raise ValueError(f"No events found for tournament slug: {tournament_slug}")
    if event_name_contains:
        target = event_name_contains.lower()
        for event in events:
            if target in (event.get("name") or "").lower():
                return event["slug"]
    if len(events) == 1:
        return events[0]["slug"]
    raise ValueError(f"Multiple events found for {tournament_slug}; set event_name_contains")

def resolve_event_from_url(url_or_slug: str, event_name_contains: Optional[str] = None) -> str:
    path = extract_startgg_path(url_or_slug)
    if is_event_slug(path):
        return path
    tournament_slug = normalize_tournament_slug(path)
    return resolve_event_slug(tournament_slug, event_name_contains=event_name_contains)

def normalize_tag(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    text = str(value).strip().lower()
    if not text:
        return None
    if "|" in text:
        text = text.split("|")[-1].strip()
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text or None

def fetch_event_attendees(event_slug: str, per_page: int = 60, delay_s: float = 0.2) -> List[Dict]:
    query = """
    query EventEntrants($slug: String, $page: Int, $perPage: Int) {
      event(slug: $slug) {
        id
        name
        entrants(query: {page: $page, perPage: $perPage}) {
          pageInfo {
            total
            totalPages
            page
            perPage
          }
          nodes {
            id
            name
            participants {
              id
              gamerTag
              prefix
              user {
                id
                slug
                authorizations(types: [DISCORD, TWITTER, TWITCH]) {
                  type
                  externalUsername
                }
              }
            }
          }
        }
      }
    }
    """
    results: List[Dict] = []
    page = 1
    total_pages = 1
    while page <= total_pages:
        data = gql(query, {"slug": event_slug, "page": page, "perPage": per_page})
        event = data.get("data", {}).get("event")
        if not event:
            raise ValueError(f"Event not found: {event_slug}")
        entrants = (event.get("entrants") or {}).get("nodes", [])
        page_info = (event.get("entrants") or {}).get("pageInfo", {})
        total_pages = page_info.get("totalPages") or 1
        for entrant in entrants:
            entrant_id = entrant.get("id")
            entrant_name = entrant.get("name")
            entrant_id_sha256 = (
                hashlib.sha256(str(entrant_id).encode("utf-8")).hexdigest()
                if entrant_id is not None
                else None
            )
            assignment = (int(entrant_id_sha256, 16) % 3) if entrant_id_sha256 is not None else None
            for participant in (entrant.get("participants") or []):
                user = participant.get("user") or {}
                auths = {
                    auth.get("type"): auth.get("externalUsername")
                    for auth in (user.get("authorizations") or [])
                }
                record = {
                    "event_slug": event_slug,
                    "event_name": event.get("name"),
                    "entrant_id": entrant_id,
                    "entrant_id_sha256": entrant_id_sha256,
                    "assignment": assignment,
                    "entrant_name": entrant_name,
                    "participant_id": participant.get("id"),
                    "gamer_tag": participant.get("gamerTag"),
                    "prefix": participant.get("prefix"),
                    "user_id": user.get("id"),
                    "user_slug": user.get("slug"),
                    "discord": auths.get("DISCORD"),
                    "twitter": auths.get("TWITTER"),
                    "twitch": auths.get("TWITCH"),
                    "youtube": None,
                }
                results.append(record)
        page += 1
        time.sleep(delay_s)
    return results

def build_attendee_df(urls: Iterable[str], event_name_contains: Optional[str] = None) -> pd.DataFrame:
    rows: List[Dict] = []
    for url in urls:
        event_slug = resolve_event_from_url(url, event_name_contains=event_name_contains)
        rows.extend(fetch_event_attendees(event_slug))
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["norm_gamer_tag"] = df["gamer_tag"].apply(normalize_tag)
    df["norm_entrant_name"] = df["entrant_name"].apply(normalize_tag)
    def row_key(row: pd.Series) -> Optional[str]:
        if pd.notna(row.get("user_id")):
            return f"user:{row['user_id']}"
        if pd.notna(row.get("user_slug")):
            return f"user_slug:{row['user_slug']}"
        if pd.notna(row.get("discord")):
            return f"discord:{str(row['discord']).lower()}"
        if pd.notna(row.get("twitter")):
            return f"twitter:{str(row['twitter']).lower()}"
        if pd.notna(row.get("twitch")):
            return f"twitch:{str(row['twitch']).lower()}"
        if pd.notna(row.get("norm_gamer_tag")):
            return f"tag:{row['norm_gamer_tag']}"
        if pd.notna(row.get("norm_entrant_name")):
            return f"entrant:{row['norm_entrant_name']}"
        return None
    df["key"] = df.apply(row_key, axis=1)
    return df

In [3]:
previous_tournaments = [
    # ngpr
    'https://www.start.gg/tournament/new-game-plus-revival-9-11',
    'https://www.start.gg/tournament/new-game-plus-revival-9-12',
    'https://www.start.gg/tournament/new-game-plus-revival-9-13',
    'https://www.start.gg/tournament/new-game-plus-revival-9-14',
    'https://www.start.gg/tournament/new-game-plus-revival-9-15',
    # allston allstars
    'https://www.start.gg/tournament/allston-allstars-vi-1',
    'https://www.start.gg/tournament/allston-allstars-v-tom-s-birthday-bash',
]
new_tournament = 'https://www.start.gg/tournament/allston-allstars-vi-1'

In [4]:
event_name_contains = "Melee Singles"  # Update if your target event uses a different name.

previous_df = build_attendee_df(previous_tournaments, event_name_contains=event_name_contains)
new_df = build_attendee_df([new_tournament], event_name_contains=event_name_contains)

if previous_df.empty:
    raise ValueError("No attendees found for previous tournaments")
if new_df.empty:
    raise ValueError("No attendees found for new tournament")

new_user_ids = set(new_df["user_id"].dropna().unique())
prospective_df = previous_df[~previous_df["user_id"].isin(new_user_ids)].copy()

prospective_df = prospective_df.drop_duplicates(subset=["user_id", "gamer_tag", "entrant_name"])
prospective_df = prospective_df.sort_values(["gamer_tag", "entrant_name"]).reset_index(drop=True)

prospective_df[
    [
        "assignment",
        "gamer_tag",
        "prefix",
        "entrant_name",
        "discord",
        "twitter",
        "twitch",
        "youtube",
    ]
]

,assignment,gamer_tag,prefix,entrant_name,discord,twitter,twitch,youtube
0,1,Andrew,None,Andrew,Padna#3980,None,None,None
1,0,Arty,HoG,HoG | Arty,Arty#5971,HoG_Arty,HoG_Arty,None
2,0,BBron,BU,BU | BBron,None,None,None,None
3,2,Babs,hc,hc | Babs,babs#5612,None,smokingrockwithmyangel,None
4,0,Bank,$G|MP,$G|MP | Bank,None,None,ssbmbank,None
...,...,...,...,...,...,...,...,...
95,1,viole,,viole,None,None,None,None
96,1,wons,PFC,PFC | wons,vwunsie,None,None,None
97,1,wons,so-po,so-po | wons,vwunsie,None,None,None
98,2,wub,None,wub,conwub#1971,None,None,None


In [5]:
intersection_user_ids = set(previous_df["user_id"].dropna().unique()) & set(new_df["user_id"].dropna().unique())
pd.DataFrame({"user_id": sorted(intersection_user_ids)})

,user_id
0,409
1,7545
2,12545
3,17294
4,30419
5,34096
6,39824
7,43464
8,45427
9,95586


In [6]:
prospective_df[
    [
        "assignment",
        "gamer_tag",
        "prefix",
        "entrant_name",
        "discord",
        "twitter",
        "twitch",
        "youtube",
    ]
].to_csv("prospective_attendees.csv", index=False)

In [10]:
prospective_df.query('gamer_tag == "Essence"')

,event_slug,event_name,entrant_id,entrant_id_sha256,assignment,entrant_name,participant_id,gamer_tag,prefix,user_id,user_slug,discord,twitter,twitch,youtube,norm_gamer_tag,norm_entrant_name,key
26,tournament/allston-allstars-v-tom-s-birthday-b...,Melee Singles,20587334,f76c7f44019c2d0107bd1196b6c5c464c01c4718ab1424...,2,Essence,18953928,Essence,None,2336818.0,user/b2af438e,yeetcheng,None,None,None,essence,essence,user:2336818.0


In [8]:
new_df

,event_slug,event_name,entrant_id,entrant_id_sha256,assignment,entrant_name,participant_id,gamer_tag,prefix,user_id,user_slug,discord,twitter,twitch,youtube,norm_gamer_tag,norm_entrant_name,key
0,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22764017,08ef2c4e7b2059d9917247b3437dd7b0cae009d1efa2ab...,1,hc | keke,20916154,keke,hc,34096,user/4864af7d,None,None,None,None,keke,keke,user:34096
1,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22762621,7074ec149bd4394334c13535a0e9e0892f952823317676...,0,MEAT,20914981,MEAT,,177939,user/abbe7832,None,None,None,None,meat,meat,user:177939
2,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22759957,a3997d0b5be6f565a8aea898d4459831a7802ce0506b88...,0,Frostie,20912511,Frostie,,3119896,user/3abaa55a,frostyshart,None,None,None,frostie,frostie,user:3119896
3,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22759640,17e80db5c69d95de2983bc3398004e9c56895518a95668...,0,hc | chery*,20912199,chery*,hc,465668,user/396ed5a7,vers_fn,None,None,None,chery,chery,user:465668
4,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22759358,c917d15a1b9089f137feb43970e82200117fdff7bc75d7...,2,Danimals,20911945,Danimals,None,43464,user/5037fe60,None,None,None,None,danimals,danimals,user:43464
5,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22758685,99603a0e6edcf996c018c61b443377a16e560d7f1b9a8d...,1,BUG | Mimes,20911415,Mimes,BUG,2549192,user/4310c026,None,None,None,None,mimes,mimes,user:2549192
6,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22758484,27e13c6a81ab37793f7f4d8831ab09a74ba7d5def9b2f3...,1,Ant,20911278,Ant,None,7545,user/36e92bc4,antssbm,YUNGTRINITRON,AntMelee,None,ant,ant,user:7545
7,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22757472,9532e77efc9dd8caaf49f41f28ab0a85014a5cc7649a0e...,2,joe chemo,20910453,joe chemo,,701249,user/18d752ee,jesusgodmatt,None,None,None,joechemo,joechemo,user:701249
8,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22756181,c950bfc630752b79e431dafdb3106f4d3ab7a2e1886606...,0,OUG | Electroman,20909235,Electroman,OUG,149864,user/11625314,Electroman#3030,Electroman_DrAl,ElectromanSSB,None,electroman,electroman,user:149864
9,tournament/allston-allstars-vi-1/event/melee-s...,Melee Singles,22755495,0d13c4184cfcad896bccb72bf5bd9c0642b7dfd2929261...,2,hc | pluto,20908601,pluto,hc,717730,user/dabc0208,None,None,None,None,pluto,pluto,user:717730
